In [2]:
! pip install xgboost

  Using cached xgboost-3.1.2-py3-none-win_amd64.whl.metadata (2.1 kB)
   ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/72.0 MB ? eta -:--:--
   ---------------------------------------- 0.8/72.0 MB 1.5 MB/s eta 0:00:47
   ---------------------------------------- 0.8/72.0 MB 1.5 MB/s eta 0:00:47
    --------------------------------------- 1.3/72.0 MB 1.4 MB/s eta 0:00:51
    --------------------------------------- 1.6/72.0 MB 1.4 MB/s eta 0:00:52
    --------------------------------------- 1.6/72.0 MB 1.4 MB/s eta 0:00:52
   - -------------------------------------- 1.8/72.0 MB 1.2 MB/s eta 0:01:01
   - -------------------------------------- 1.8/72.0 MB 1.2 MB/s eta 0:01:01
   - -------------------------------------- 2.1/72.0 MB 1.1 MB/s eta 0:01:03
   - -------------------------------------- 2.1/72.0 MB 1.1 MB/s eta 0:01:03
   - ----------------------

In [23]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
from sklearn.preprocessing import OneHotEncoder

In [25]:
df = pd.read_csv("/content/Enhanced_Data.csv")

selected_cols = [
    "Quantity", "Discount", "Profit", "Shipping Cost", "Oil Price",
    "Year", "Month", "DayOfWeek",
    "Category", "Sub-Category", "Ship Mode", "Market", "Region", "Segment", "Season",
    "Sales"
]
df_selected = df[selected_cols].copy()

# ✅ نحول Sales إلى أرقام لو كانت مكتوبة كنص
df_selected['Sales'] = pd.to_numeric(df_selected['Sales'], errors='coerce')

cat_cols = df_selected.select_dtypes(include=['object']).columns
num_cols = df_selected.select_dtypes(include=['int64','float64']).columns
num_cols = [col for col in num_cols if col != 'Sales']  # ✅ الطريقة الآمنة بدل drop()

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoded_data = encoder.fit_transform(df_selected[cat_cols])
encoded_df = pd.DataFrame(encoded_data, columns=encoder.get_feature_names_out(cat_cols), index=df_selected.index)

# دمج الأعمدة
final_df = pd.concat([df_selected[num_cols], encoded_df], axis=1)

# الهدف
y = df_selected['Sales']

# تنظيف القيم
final_df = final_df.replace([np.inf, -np.inf], np.nan).fillna(0)
y = y.replace([np.inf, -np.inf], np.nan).fillna(0)

In [26]:
print(df.columns.tolist())

['Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'City', 'State', 'Country', 'Market', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit', 'Shipping Cost', 'Order Priority', 'Season', 'Is Promotion', 'Discount Level', 'Oil Price', 'Year', 'Month', 'Week', 'Day', 'DayOfWeek', 'DayName', 'DayOfYear']


In [27]:
X = final_df
y = df['Sales']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [28]:
model = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [30]:
# Evaluation
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

y_pred = model.predict(X_test)

rmse = mean_squared_error(y_test, y_pred) ** 0.5  # ✅ حساب RMSE يدويًا
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"R2 Score: {r2:.4f}")

RMSE: 177.5490
MAE: 66.5110
R2 Score: 0.8609
